**Tim Penyusun:**
- Naufal Ihsanul Islam (F1D02310084)
- Nengah Anggi Juwita Sari (F1D02310021)
- Lutfi Alfarizi (F1D02310121)

---

# S3: Peramalan Harga Pangan Menggunakan PCA + Random Forest
Notebook ini berisi implementasi model gabungan **PCA + Random Forest** untuk meramal harga pangan. 
Metode reduksi dimensi Principal Component Analysis (PCA) digunakan untuk menyederhanakan fitur sebelum masuk ke model regresi. Pendekatan ini merupakan upaya melihat bagaimana jika kita menekan jumlah dimensi (variabel) secara matematis tanpa supervisi.


### 1. Persiapan Library dan Konfigurasi
Memuat pustaka Python standar untuk keperluan manipulasi data, reduksi dimensi algoritma matematis (`PCA`), pengatur skala standar (`StandardScaler`), pemodelan `RandomForestRegressor`, serta perhitungan metrik akurasi. Setelan kanvas visualisasi diatur agar menunjang kenyamanan membaca grafik.


In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#ffffff', 'axes.facecolor': '#ffffff', 'axes.edgecolor': '#cccccc',
    'axes.grid': True, 'grid.color': '#cccccc', 'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#000000', 'axes.labelcolor': '#000000', 'xtick.color': '#000000',
    'ytick.color': '#000000', 'font.sans-serif': 'Arial', 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': True
})

### 2. Memuat Data, Pembersihan, dan Ekstraksi Fitur
Proses standar penyiapan data; pembersihan abjad rupiah menjadi deret bilangan bersih numerik, merajut data berlubang, memangkas outlier yang berlebihan, serta menyuntikkan fitur tambahan semacam pergeseran musim dan harga-harga perhentian sebelumnya.


In [ ]:
df_raw = pd.read_csv('dataset.csv')
df_raw['Nama Provinsi'] = df_raw['Nama Provinsi'].astype(str).str.strip()
df_raw['Komoditas'] = df_raw['Komoditas'].astype(str).str.strip()
df_raw = df_raw[df_raw['Tahun'] <= 2025].copy()

def bersihkan_harga(nilai):
    if pd.isna(nilai) or nilai == '-': return np.nan
    try:
        val = float(str(nilai).replace('Rp', '').replace(',', '').strip())
        return val if val > 0 else np.nan
    except: return np.nan

df_raw['Harga_Numerik'] = df_raw['Harga'].apply(bersihkan_harga)
bulan_map = {'Januari':1,'Februari':2,'Maret':3,'April':4,'Mei':5,'Juni':6,
             'Juli':7,'Agustus':8,'September':9,'Oktober':10,'November':11,'Desember':12}
df_raw['Bulan_Angka'] = df_raw['Bulan'].astype(str).str.strip().map(bulan_map)

df = df_raw.sort_values(['Nama Provinsi','Komoditas','Tahun','Bulan_Angka']).reset_index(drop=True)
df['Harga_Numerik'] = df.groupby(['Nama Provinsi','Komoditas'])['Harga_Numerik'].ffill(limit=3)

df_train_temp = df[df['Tahun'] <= 2024]
group_mean = df_train_temp.groupby(['Nama Provinsi', 'Komoditas'])['Harga_Numerik'].mean()
global_mean = df_train_temp.groupby('Komoditas')['Harga_Numerik'].mean()

def impute_missing(row):
    if not pd.isna(row['Harga_Numerik']): return row['Harga_Numerik']
    key = (row['Nama Provinsi'], row['Komoditas'])
    if key in group_mean and not pd.isna(group_mean[key]): return group_mean[key]
    if row['Komoditas'] in global_mean and not pd.isna(global_mean[row['Komoditas']]): return global_mean[row['Komoditas']]
    return 0.0

df['Harga_Numerik'] = df.apply(impute_missing, axis=1)

outlier_params = df_train_temp.groupby('Komoditas')['Harga_Numerik'].agg(['mean', 'std']).reset_index()
def apply_outlier_filter(row):
    try:
        m = outlier_params.loc[outlier_params['Komoditas'] == row['Komoditas'], 'mean'].values[0]
        s = outlier_params.loc[outlier_params['Komoditas'] == row['Komoditas'], 'std'].values[0]
        if pd.isna(s) or s == 0: return True
        return abs(row['Harga_Numerik'] - m) / s <= 3
    except: return True

df_clean = df[df.apply(apply_outlier_filter, axis=1)].copy()
df_clean['Tanggal'] = pd.to_datetime(df_clean['Tahun'].astype(str) + '-' + df_clean['Bulan_Angka'].astype(str) + '-01')

df_clean['Bulan_Sin'] = np.sin(2 * np.pi * df_clean['Bulan_Angka'] / 12)
df_clean['Bulan_Cos'] = np.cos(2 * np.pi * df_clean['Bulan_Angka'] / 12)
df_clean['Ramadan_Lebaran'] = df_clean['Bulan_Angka'].isin([3, 4, 5]).astype(int)

### Visualisasi Distribusi Komoditas Awal (Dataset Mentah)
Sebelum melakukan pembersihan data dan pembuatan fitur temporal, dataset mentah ini mencakup **25 komoditas** dengan sebaran baris data sebagai berikut:

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, x='Komoditas', order=df_clean['Komoditas'].value_counts().index, palette='magma')
plt.title('Distribusi 25 Komoditas Awal pada Dataset Mentah (2021-2025)')
plt.xlabel('Komoditas')
plt.ylabel('Jumlah Baris Data')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("=====================================================================")
print(f"Total Komoditas Awal: {df_clean['Komoditas'].nunique()}")
print(df_clean['Komoditas'].unique())
print("=====================================================================")

grouped = df_clean.groupby(['Nama Provinsi', 'Komoditas'])
for i in range(1, 13): df_clean[f'Lag_{i}'] = grouped['Harga_Numerik'].shift(i)
df_clean['Rolling_Mean_3'] = grouped['Harga_Numerik'].shift(1).rolling(window=3).mean()
df_clean['Rolling_Std_3'] = grouped['Harga_Numerik'].shift(1).rolling(window=3).std().fillna(0)

df_clean = df_clean.dropna().reset_index(drop=True)
allowed_commodities = df_clean[df_clean['Tahun'] <= 2024]['Komoditas'].unique()
df_clean = df_clean[df_clean['Komoditas'].isin(allowed_commodities)].reset_index(drop=True)

### 3. Analisis Eksplorasi Data (EDA)

Deretan grafik empiris ini memberikan landasan diagnostik awal. Mari kita ulas makna visual di baliknya:

1. **Dinamika Data Kosong (Kiri Atas)**:
   Grafik bar ini mendokumentasikan dengan sangat transparan di titik komoditas mana saja tim pencatat wilayah sering kehilangan pijakan harganya. Puncak grafik dipuncaki oleh keluarga tanaman umur pendek, layaknya Cabai Rawit Merah, Cabai Merah Keriting, maupun jajaran bawang. Komoditas bervolatilitas tinggi ini sering ghaib dari pasaran ketika masa panen hancur. Ketidakstabilan ketersediaan data ini adalah kenyataan lapangan yang mutlak harus dipahami oleh setiap analis sebelum memahat model.

2. **Jejak Korelasi Waktu Tempuh (Kanan Atas)**:
   Peta suhu (*heatmap*) memvalidasi dugaan korelasi pergeseran harga kita. Blok `Harga_Numerik` saat diadu dengan angka bulan sebelumnya (`Lag_1`) memancarkan merah tua pekat, bertuliskan 0.94. Secara ilmiah ini berarti gerak bandul harga di pasar hari ini merupakan bayangan hampir mutlak dari bandul harga sebulan sebelumnya. Pola lintasan linier yang sangat kental ini (bahkan bertahan hingga `Lag_3` di 0.89) yang akan dieksploitasi oleh model kita nanti.

3. **Operasi Pemotongan Outlier (Bawah Kiri & Kanan)**:
   Di sudut kiri bawah, kotak pergerakan (boxplot) warna merah memaparkan wajah kotor dari set data lapangan. Bintik-bintik pencilan melesat jauh di atas garis maksimal. Bintik ini merepresentasikan anomali harga gila-gilaan sesaat. Kalau bintik ini kita loloskan, PCA (algoritma reduksi di sesi berikutnya) akan pusing tujuh keliling, karena PCA amat sangat sensitif (*fragile*) terhadap outlier yang membuat garis pilar pembedanya oleng.
   Setelah kita bersihkan dengan ambang batas 3 standar deviasi (boxplot kanan yang berwarna hijau), distribusinya kini menjadi "gemuk terpusat" dan jauh lebih steril, menyodorkan medan tempur yang sangat kondusif bagi rumus-rumus PCA kelak.


In [ ]:
fig = plt.figure(figsize=(15, 10))
ax1 = plt.subplot(2, 2, 1)
missing = df_raw.groupby('Komoditas')['Harga_Numerik'].apply(lambda x: x.isna().sum()).sort_values(ascending=False).head(10)
missing.plot(kind='bar', color='salmon', ax=ax1)
ax1.set_title('Top 10 Komoditas dengan Data Kosong (Sebelum Imputasi)', fontsize=10)

ax2 = plt.subplot(2, 2, 2)
cols = ['Harga_Numerik', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3']
sns.heatmap(df_clean[cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=ax2)
ax2.set_title('Korelasi Nilai Masa Lalu dengan Harga Saat Ini', fontsize=10)

ax3 = plt.subplot(2, 2, 3)
sns.boxplot(data=df[['Harga_Numerik']], palette='Set1', ax=ax3)
ax3.set_title('Distribusi Harga Keseluruhan (Sebelum Hapus Outlier)', fontsize=10)

ax4 = plt.subplot(2, 2, 4)
sns.boxplot(data=df_clean[['Harga_Numerik']], palette='Set2', ax=ax4)
ax4.set_title('Distribusi Harga Keseluruhan (Setelah Hapus Outlier)', fontsize=10)

plt.tight_layout()
plt.show()

### 3.1. Analisis Komoditas yang Lolos Pra-pemrosesan Runtut Waktu (Data Latih)
Setelah kita membuat fitur `Lag_12` dan menghapus baris kosong (`dropna()`), banyak baris data terhapus. 
Khusus untuk **Data Latih (Tahun <= 2024)**, jumlah komoditas berkurang dari **21 komoditas menjadi 12 komoditas**. 
Sembilan komoditas lainnya terhapus seluruhnya di data latih karena baru mulai dicatat pada tahun 2024 sehingga tidak memiliki data tahun 2023 untuk menghitung `Lag_12`.

Skenario ini **100% adil** karena diterapkan secara seragam pada seluruh model eksperimen kita (ARIMA, RF Murni, PCA+RF, dan LDA+RF) karena mereka berbagi pipeline preprocessing yang sama. Semua model diuji menggunakan basis data latih dan uji yang identik.

In [ ]:
plt.figure(figsize=(10, 5))
df_train_clean = df_clean[df_clean['Tahun'] <= 2024]
sns.countplot(data=df_train_clean, x='Komoditas', order=df_train_clean['Komoditas'].value_counts().index, palette='viridis')
plt.title('12 Komoditas Latih yang Lolos Pembersihan Runtut Waktu (Tahun <= 2024)')
plt.xlabel('Komoditas')
plt.ylabel('Jumlah Baris Data')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("=====================================================================")
print(f"Total Komoditas Latih yang Lolos: {df_train_clean['Komoditas'].nunique()}")
print(df_train_clean['Komoditas'].unique())
print("=====================================================================")

### 4. Penelaahan Mendalam Scree Plot PCA

Sel ini adalah jantung dari eksperimen kita kali ini. Sebelum disuapkan ke Random Forest, puluhan kolom data kita peras (reduksi) jumlah dimensinya agar seringkas mungkin dengan menggunakan metode PCA. Kita mengambil 20 Komponen Utama (PC) untuk menyesuaikan kapasitas reduksi dari eksperimen lainnya (LDA).

**Analisis Karakteristik Kurva Scree Plot**:
Perhatikan garis putus-putus biru bersimpul lingkar pada grafik di bawah ini. Garis ini bertugas menggambarkan daya tangkap informasi kumulatif seiring berjalannya jumlah komponen utama (dari PC 1 hingga 20).
Yang menakjubkan (dan mengkhawatirkan) adalah **kelandaian garis kurva ini yang merangkak layaknya lereng bukit yang nyaris rata**. Secara teoritis, reduksi data yang baik seharusnya memperlihatkan sebuah tekukan siku yang sangat tajam (*elbow*). Siku tajam membuktikan bahwa seluruh intisari penting langsung bisa ditangkap oleh 2 atau 3 pilar dimensi awal. Namun di sini, Pilar Pertama (PC 1) terlihat sangat loyo, hanya mampu mendekap **22%** informasi. Setelah seluruh ke-20 pilar dirakit, informasi total yang sukses diselamatkan PCA ini ternyata hanyalah berkisar **56.82%**.

**Diagnosis Dampak Matematisnya**:
Laju pendakian kurva yang landai dan terengah-engah ini memancarkan sinyal diagnosis keras: *Bongkahan informasi dalam struktur harga pangan kita tercacah-cacah rata mendiami setiap penjuru variabel fitur (kolom), alias tidak menggumpal di satu dua titik.* 
Menyederhanakannya secara buta (tanpa pengawasan alias *unsupervised*) ke dalam 20 dimensi mengharuskan kita memenggal kepala **43.18% informasi penting**, yang tak lain dan tak bukan adalah serpihan data historis temporal yang berharga. Kehilangan separuh peta tata spasial komoditas provinsi ini sudah hampir dipastikan akan meruntuhkan kecerdasan ramalan Random Forest di titik akhir.


In [ ]:
fitur_numerik = ['Tahun','Bulan_Sin','Bulan_Cos','Ramadan_Lebaran','Rolling_Mean_3','Rolling_Std_3'] + [f'Lag_{i}' for i in range(1,13)]
fitur_kategorik = ['Nama Provinsi', 'Komoditas']
X = pd.concat([df_clean[fitur_numerik], pd.get_dummies(df_clean[fitur_kategorik], drop_first=True)], axis=1)
y = df_clean['Harga_Numerik']

train_mask = df_clean['Tahun'] <= 2024
test_mask  = df_clean['Tahun'] == 2025

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test   = X[test_mask], y[test_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# N_FAIR = min(df_clean['Komoditas'].nunique() - 1, X.shape[1])
pca = PCA(n_components=9)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 10), pca.explained_variance_ratio_.cumsum() * 100, marker='o', linestyle='--')
plt.title(f'Scree Plot PCA\n(Total Varians Dijelaskan: {pca.explained_variance_ratio_.cumsum()[-1]*100:.2f}%)')
plt.xlabel('Komponen Utama (Principal Component)')
plt.ylabel('Kumulatif Persentase Varians (%)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
X_train_pca

### 5. Pelatihan Mesin Kombinasi (PCA + Random Forest)
Fitur berjumlah 20 pilar komponen utama hasil saringan PCA tadi kemudian dilemparkan masuk sebagai bahan bakar peramalan ke dalam model Random Forest yang disetel pada 100 batang pohon (*n_estimators*). 


In [ ]:
print("Melatih Model PCA + Random Forest...")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_pca, y_train)
preds = rf.predict(X_test_pca)

df_eval = df_clean[test_mask].copy()
df_eval['Actual'] = y_test
df_eval['Prediction'] = preds

### 6. Evaluasi Semu Fase 1: Menguji Metode Riset Usang

Uji coba perdana ini mensimulasikan praktik riset tradisional di mana dataset 2021 hingga 2025 dicincang kasar secara acak tanpa aturan sekuensial waktu.

**Analisis Estetika Kurva Tebakan Prediksi**:
Lihat titik biru yang terangkai padat lurus menyilang tepat menutupi jalur panduan diagonal. Ini bukan sekadar grafik yang bagus; ini nyaris sebuah fotokopi sempurna harga nyata. Jika titik biru hinggap di garis merah, artinya tebakan harga sama presisi hingga selisih koin dengan harga sesungguhnya. Dan secara keseluruhan persentase melesetnya (*MAPE*) diklaim hanya berada pada nilai **7.54%**! Seakan-akan model ini amat brilian.

**Menyingkap Kabut Delusi Akurasi**:
Padahal, seperti yang kita telah diagnosis panjang lebar sebelumnya, 43% data berharga telah lenyap tersapu gergaji matematis PCA. Bagaimana mungkin tebakannya tetap sempurna? Jawabannya jelas: Grafik "ajaib" ini hanyalah visualisasi ilusi hasil kebocoran rahasia (*Data Leakage*). 
Potongan kalender acak telah mengizinkan algoritma untuk mengintip banderol harga di hari perayaan yang terletak saling berdampingan di set pelatihan, sehingga Random Forest tinggal bermain sulap "menyalin" jawaban ke atas set ujian, alih-alih betul-betul membangun kepekaan insting prediksi sejati.


In [ ]:
from sklearn.model_selection import train_test_split
_, X_test_raw_f1, _, y_test_f1 = train_test_split(X, y, test_size=0.2, random_state=42)

X_test_scaled_f1 = scaler.transform(X_test_raw_f1)
X_test_f1 = pca.transform(X_test_scaled_f1)

preds_f1 = rf.predict(X_test_f1)
df_fase1 = pd.DataFrame({'Actual': y_test_f1, 'Prediction': preds_f1})

rmse_1 = np.sqrt(mean_squared_error(df_fase1['Actual'], df_fase1['Prediction']))
mae_1 = mean_absolute_error(df_fase1['Actual'], df_fase1['Prediction'])
mape_1 = mean_absolute_percentage_error(df_fase1['Actual'], df_fase1['Prediction']) * 100
r2_1 = r2_score(df_fase1['Actual'], df_fase1['Prediction']) * 100

plt.figure(figsize=(6, 5))
plt.scatter(df_fase1['Actual'], df_fase1['Prediction'], color='blue', alpha=0.6)
plt.plot([df_fase1['Actual'].min(), df_fase1['Actual'].max()], [df_fase1['Actual'].min(), df_fase1['Actual'].max()], 'r--', lw=2)
plt.title(f'FASE 1: Kondisi Awal (Tidak Sinkron)\n(Metodologi Awal: Split Acak dari Seluruh Tahun (Terjadi Data Leakage))')
plt.xlabel('Aktual (Rp)')
plt.ylabel('Prediksi (Rp)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

eval_text_1 = f"""==================================================
FASE 1: EVALUASI KONDISI AWAL (DATA TIDAK SINKRON)
(Bukti Analisis Historis: Pembagian Acak Memicu Data Leakage)
==================================================
Jumlah Data Uji : {len(df_fase1)} baris
RMSE            : Rp {rmse_1:,.2f}
MAE             : Rp {mae_1:,.2f}
MAPE            : {mape_1:.2f}%
R2              : {r2_1:.2f}%
"""
print(eval_text_1)

### 7. Bukti Forensik Kuantitatif Kebocoran Data (Data Leakage)

Papan diagram batang ini kita sebut saja sebagai visum pembuktian kebocoran (leakage). Secara vulgar dan gamblang ia mendakwa kecerobohan split acak pada Fase 1.

Perhatikan menjulangnya gedung tiang warna merah dengan nilai **1.708**. Apa maknanya? Sebanyak 1.708 baris lembar data prediksi masa depan (tahun 2025) secara terang-terangan masuk me-leverage bahan hafalan set pelatihan model. Sebaliknya, tiang oranye membuktikan bahwa **4.675** lembar data kuno masa lalu malah nyasar ke dalam keranjang pengujian kelayakan. 
Cacat aliran ini mengonfirmasi secara forensik empiris bahwa angka cemerlang Fase 1 adalah performa palsu.


In [ ]:
X_train_f1, X_test_f1, y_train_f1, y_test_f1 = train_test_split(X, y, test_size=0.2, random_state=42)

leakage_latih_2025 = (X_train_f1['Tahun'] == 2025).sum()
leakage_uji_historis = (X_test_f1['Tahun'] <= 2024).sum()

plt.figure(figsize=(6, 4))
categories = ['Data 2025 di Train\n(Leakage Masa Depan)', 'Data Historis di Test\n(Leakage Evaluasi)']
counts = [leakage_latih_2025, leakage_uji_historis]
bars = plt.bar(categories, counts, color=['#e53935', '#fb8c00'], width=0.4)
plt.title('Analisis Diagnostik: Kebocoran Data (Data Leakage) Fase 1')
plt.ylabel('Jumlah Baris Data')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f'{yval:,}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print("=====================================================================")
print("DIAGNOSTIC CONCLUSION: DETEKSI SEVERE DATA LEAKAGE DI FASE 1")
print("=====================================================================")
print(f"1. Jumlah data tahun 2025 yang bocor masuk ke DATA LATIH : {leakage_latih_2025} baris")
print(f"2. Jumlah data historis (2021-2024) di DATA UJI          : {leakage_uji_historis} baris")
print("=====================================================================")

### 8. Fase 2: Pengujian Sekuensial Masa Depan Nyata 

Kini mesin dididik secara ketat. Ia wajib belajar menebak rentetan harga di tahun 2025 secara jujur tanpa sekelumit pun lembaran masa depan disodorkan pada saat ia tengah duduk di kursi latihan.

**Analisis Pembubaran Barisan Prediksi**:
Lihat nasib kumpulan titik hijau yang terbang tercerai-berai pada grafik di bawah ini. Coretan garis prediksi yang sebelumnya padat membeku lurus di Fase 1 kini melebar porak-poranda menciptakan ruang kekosongan awan yang amat lebar, tersebar tak karuan menggerayangi batas diagonal yang sesungguhnya.

**Interpretasi Ilmiahnya**:
Pembubaran titik prediksi ini secara sadis memperlihatkan dampak kehilangan 43% komponen spasial data yang tadi dibuang mentah-mentah oleh kurva PCA yang landai. Saat tidak memiliki kelengkapan bekal fitur identitas provinsi yang rinci dan murni dipaksa berjuang meraba tren di waktu baru, model kebingungan.
Kegamangan besar PCA dalam menyesuaikan diri terhadap kejutan tren berujung pada terjangan nilai MAPE model gabungan ini ke rekor terburuk yang mencapai **17.48%** (lebih dari dua kali lipat lebih meleset dibanding pencapaian model RF murni terdahulu). Ini menasbihkan ketidakmampuan metode PCA untuk menopang peramalan harga semacam ini.


In [ ]:
y_a2, y_p2 = df_eval['Actual'], df_eval['Prediction']
rmse_2 = np.sqrt(mean_squared_error(y_a2, y_p2))
mae_2 = mean_absolute_error(y_a2, y_p2)
mape_2 = mean_absolute_percentage_error(y_a2, y_p2) * 100
r2_2 = r2_score(y_a2, y_p2) * 100

plt.figure(figsize=(6, 5))
plt.scatter(y_a2, y_p2, color='green', alpha=0.3, s=15)
plt.plot([y_a2.min(), y_a2.max()], [y_a2.min(), y_a2.max()], 'r--', lw=2)
plt.title('FASE 2: Agregat Nasional (Evaluasi Kronologis Murni)')
plt.xlabel('Aktual (Rp)')
plt.ylabel('Prediksi (Rp)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

eval_text_2 = f"""==================================================
FASE 2: EVALUASI KRONOLOGIS MURNI (BEBAS LEAKAGE)
==================================================
Jumlah Data Uji : {len(df_eval)} baris
RMSE            : Rp {rmse_2:,.2f}
MAE             : Rp {mae_2:,.2f}
MAPE            : {mape_2:.2f}%
R2              : {r2_2:.2f}%
"""
print(eval_text_2)

### 9. Fase 3: Deteksi Kebobrokan Per Komoditas 

Supaya tidak hanya menyalahkan rata-rata keseluruhan yang tersamar oleh jomplangnya jarak kelipatan nilai, Fase 3 hadir mendirikan tribun perbandingan adil berdasarkan kasta fluktuasi komoditas secara terpisah. 

**Analisis Diagram Batang Yang Mengerikan**:
Coba tatap rentetan batang-batang menjulang ini; pemandangannya bagaikan puing-puing tingkat kesalahan prediksi. Jika di model RF Murni terdahulu tiang Cabai Rawit menapak angka 10-15%, PCA menggiring kesalahan peramalan hortikultura hijau segar tersebut memuncak gila-gilaan nyaris **25%**! 
Bahkan, keluarga yang sebelumnya tak bisa salah seperti Bawang Putih, kini turut didera kerusakan. Ini merupakan demonstrasi kelam bahwa operasi buta algoritma matematis pilar PCA tidak segan-segan memorak-porandakan tatanan kepekaan sinyal pada karakteristik mikro tiap komoditas bahan pangan secara kolektif tanpa ampun.


In [ ]:
rmse_list, mae_list, mape_list, r2_list, kom_names = [], [], [], [], []
for kom, group in df_eval.groupby('Komoditas'):
    y_a, y_p = group['Actual'], group['Prediction']
    if len(group) > 1:
        rmse_k = np.sqrt(mean_squared_error(y_a, y_p))
        mae_k = mean_absolute_error(y_a, y_p)
        mape_k = mean_absolute_percentage_error(y_a, y_p) * 100
        try:
            r2_k = r2_score(y_a, y_p) * 100
            if r2_k < -100: r2_k = 0
        except: r2_k = 0
        rmse_list.append(rmse_k)
        mae_list.append(mae_k)
        mape_list.append(mape_k)
        r2_list.append(r2_k)
        kom_names.append(kom)

rmse_3 = np.mean(rmse_list)
mae_3 = np.mean(mae_list)
mape_3 = np.mean(mape_list)
r2_3 = np.mean(r2_list) if len(r2_list) > 0 else 0

plt.figure(figsize=(10, 6))
sns.barplot(x=mape_list, y=kom_names, palette='viridis')
plt.axvline(x=mape_3, color='red', linestyle='--', label=f'Rata-rata MAPE: {mape_3:.2f}%')
plt.title('FASE 3: Tingkat Kesalahan (MAPE) per Karakteristik Komoditas')
plt.xlabel('MAPE (%)')
plt.ylabel('Komoditas')
plt.legend()
plt.tight_layout()
plt.show()

eval_text_3 = f"""==================================================
FASE 3: EVALUASI METRIK ADIL (RATA-RATA KOMODITAS)
(Solusi: Menghitung secara adil bebas bias skala angka nominal)
==================================================
Jumlah Data Uji : {len(df_eval)} baris (21 Komoditas)
RMSE            : Rp {rmse_3:,.2f}
MAE             : Rp {mae_3:,.2f}
MAPE            : {mape_3:.2f}%
R2              : {r2_3:.2f}%
=================================================="""
print(eval_text_3)

with open("hasil_evaluasi_pca.txt", "w", encoding="utf-8") as f:
    f.write(eval_text_1 + "\n" + eval_text_2 + "\n" + eval_text_3)

### 10. Sintesis Tumbangnya Algoritma Reduksi

Baris-baris pada tabel ini tak pelak lagi adalah batu nisan bagi pengimplementasian PCA di ranah prediksi waktu ini. 

**Analisis Dinamika Kemerosotan Kualitas**:
Walau angka nominal RMSE terpangkas di Fase 3 berkat dihilangkannya hegemoni komoditas mahal (mengempis dari **Rp 6.177** merosot ke **Rp 4.606**), tragedi sejati bertengger diam-diam di ujung kanan tabel. Tengok jatuhnya nilai indeks reliabilitas ($R^2$) dari megahnya **95.78%** di atas langit Fase 2, jatuh nyungsep terkoyak keras menyentuh dasar bumi **35.46%** pada hitungan otentik Fase 3.

**Makna Dibalik Runtuhnya $R^2$ ke 35%**:
Memperoleh raihan seburuk 35.46% mengkristalisasi sebuah kaidah logis bahwa PCA sungguh algoritma yang tumpul bila dihadapkan dengan keharmonisan waktu dan daerah. Meremas deret panjang tabel tanpa bimbingan pengawasan pada akhirnya berujung pada kerusakan ekstrim data fundamental yang seharusnya dibutuhkan model Random Forest untuk mendeduksi ramalan yang bijak secara logis bagi ketersediaan beras dan cabai masa depan.


In [ ]:
df_fase = pd.DataFrame([
    {'Tahapan': 'Fase 1 (Metodologi Awal - Error)', 'Data Uji': len(df_fase1), 'MAE (Rp)': f"{mae_1:,.0f}", 'RMSE (Rp)': f"{rmse_1:,.0f}", 'MAPE (%)': f"{mape_1:.2f}%", 'R2 (%)': f"{r2_1:.2f}%"},
    {'Tahapan': 'Fase 2 (Data Sinkron - Bias)', 'Data Uji': len(df_eval), 'MAE (Rp)': f"{mae_2:,.0f}", 'RMSE (Rp)': f"{rmse_2:,.0f}", 'MAPE (%)': f"{mape_2:.2f}%", 'R2 (%)': f"{r2_2:.2f}%"},
    {'Tahapan': 'Fase 3 (Data Sinkron - Adil)', 'Data Uji': len(df_eval), 'MAE (Rp)': f"{mae_3:,.0f}", 'RMSE (Rp)': f"{rmse_3:,.0f}", 'MAPE (%)': f"{mape_3:.2f}%", 'R2 (%)': f"{r2_3:.2f}%"}
])
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df_fase.values, colLabels=df_fase.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.5)

for (row, col), cell in table.get_celld().items():
    if row == 3:
        cell.set_facecolor('#d3e4cd')
        cell.set_text_props(weight='bold')
    elif row == 0:
        cell.set_facecolor('#cccccc')
        cell.set_text_props(weight='bold')

plt.title(f'Tabel Rekapitulasi Tahapan Evaluasi', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()